# Lección 11.3 — Redes recurrentes (RNN) y su memoria corta

> **Módulo 11 · Secuencias y lenguaje** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 9.1 (la neurona), Lección 9.3 (tanh), Lección 9.4 (retropropagación), Lección 9.7 (gradientes que desaparecen/explotan), Lecciones 11.1 y 11.2

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía cómo una **red recurrente** lee una secuencia paso a paso con una **memoria** (estado oculto).
- Calcular **a mano** unos pasos de una RNN y reconocer la misma operación en PyTorch.
- Entrenar RNN pequeñas para **predecir una señal** y para **inventar nombres** letra por letra.
- Explicar por qué su memoria es **corta**: gradientes que se desvanecen.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams["figure.figsize"] = (9, 3.5)
device = "cuda" if torch.cuda.is_available() else "cpu"     # todo corre bien en CPU
torch.manual_seed(0)
print("✅ Todo listo · dispositivo:", device)

---
## 1. 📖 La intuición: leer con un post-it

Imagina que lees una novela, **una palabra a la vez**, y solo puedes llevar tus notas en **un post-it**. Con cada palabra nueva:
1. Lees la palabra.
2. Miras lo que tenías anotado en el post-it.
3. **Reescribes** el post-it combinando ambas cosas.

```
 palabra 1 ──►[🧠]──► post-it 1
                         │
 palabra 2 ──►[🧠]◄──────┘──► post-it 2
                                 │
 palabra 3 ──►[🧠]◄──────────────┘──► post-it 3  ──► "¿de qué trata la historia?"
```

Eso es una **red neuronal recurrente** (RNN, *Recurrent Neural Network*):
- El **post-it** es el **estado oculto** (*hidden state*), un vector de números: la memoria de la red.
- El **🧠** es **la misma** pequeña red, que se usa en cada paso.
- Al final, el último post-it **resume** toda la secuencia.

A diferencia de las ventanas de la Lección 11.1, la RNN puede leer secuencias de **cualquier longitud**, y lo que aprende sobre "una palabra" sirve para cualquier posición.

### ✋ Ejercicio 1 — El problema del post-it
El post-it tiene un tamaño fijo y lo reescribes con cada palabra. Si la novela tiene 300 páginas, ¿qué crees que pasará con lo que leíste en la página 1 cuando llegues al final?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Probablemente **se habrá borrado**. Cada vez que reescribes el post-it, lo viejo se diluye un poco para hacerle espacio a lo nuevo. Después de miles de reescrituras, del principio no queda casi nada.

Esa es exactamente la gran debilidad de las RNN: su **memoria es corta**. Lo comprobarás con números en la Sección 4.
</details>

---
## 2. 📖 El concepto: una neurona que se mira a sí misma

En cada paso *t*, la RNN calcula su nueva memoria **hₜ** a partir de la entrada **xₜ** y de su memoria anterior **hₜ₋₁**:

```
hₜ = tanh( wₓ · xₜ  +  wₕ · hₜ₋₁  +  b )
          └─ lo nuevo ─┘  └─ lo que recordaba ─┘
```

Es una neurona de la Lección 9.1 con una entrada extra: **su propia salida del paso anterior**. La `tanh` (Lección 9.3) mantiene la memoria entre −1 y 1.

- `wₓ` dice cuánto le importa **lo nuevo**.
- `wₕ` dice cuánto **conserva** de lo que recordaba.

Hagámoslo a mano con una RNN de **una sola neurona**.

In [ ]:
def rnn_a_mano(secuencia, w_x, w_h, b=0.0):
    h = 0.0                                   # el post-it empieza en blanco
    for t, x in enumerate(secuencia, start=1):
        h = np.tanh(w_x * x + w_h * h + b)
        print(f"paso {t}: entrada = {x:>4}  →  memoria h = {h:+.3f}")
    return h

rnn_a_mano([1, 0, 0, 0, 0], w_x=1.0, w_h=0.5);

### 🔮 Ejercicio 2 — ¿Cuánto recuerda?
Arriba, la red vio un **1** en el primer paso y luego solo ceros. Fíjate cómo se apaga el recuerdo.

Ahora cambiaremos `w_h` de 0.5 a **0.9**. ¿La memoria en el paso 5 será **mayor o menor** que antes? ¿Y si pones `w_h = 0`?

✍️ *Mi predicción:*

In [ ]:
W_H = 0.9        # ✏️ prueba 0, 0.5, 0.9, 1.5
rnn_a_mano([1, 0, 0, 0, 0], w_x=1.0, w_h=W_H);

<details><summary><b>👀 Ver respuesta</b></summary>

- Con `w_h = 0.5` la memoria cae a la mitad en cada paso: 0.76 → 0.36 → 0.18 → 0.09 → **0.04**.
- Con `w_h = 0.9` se apaga más despacio: en el paso 5 queda **≈0.36**.
- Con `w_h = 0` la red **no tiene memoria**: en el paso 2 ya vale 0. Solo ve la entrada actual.
- Con `w_h = 1.5`, la memoria **no se apaga**: se acerca a ≈0.86 y se queda ahí "pegada". La `tanh` evita que crezca sin límite.

`w_h` es la perilla de **cuánto conserva**. Pero fíjate: con `w_h` < 1 el recuerdo se apaga **multiplicándose** por un número menor que 1 en cada paso. Guarda esa idea para la Sección 4.
</details>

### 📖 Desenrollar la red
Como se usa **la misma** neurona en cada paso, una RNN se suele dibujar "desenrollada" en el tiempo:

```
   h₀ ──►[ W ]──► h₁ ──►[ W ]──► h₂ ──►[ W ]──► h₃ ──►[ W ]──► h₄
           ▲              ▲              ▲              ▲
           x₁             x₂             x₃             x₄
```

Los **mismos pesos W** en todas las cajas. Esto se llama **compartir parámetros** (*parameter sharing*), la misma idea que en las convoluciones (Lección 10.2): allí el mismo filtro recorre **el espacio** de la imagen; aquí la misma neurona recorre **el tiempo**.

### ✋ Ejercicio 3
1. Una RNN con 4 pasos desenrollados, ¿tiene 4 veces más pesos que con 1 paso?
2. ¿Por qué compartir pesos permite procesar frases de cualquier longitud?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **No.** Tiene exactamente los mismos pesos; solo los **usa** 4 veces.
2. Porque la red solo sabe hacer **un paso**: "combina lo nuevo con lo que recordabas". Ese paso se puede repetir 3 o 3,000 veces. Una red normal con una entrada por posición (como las ventanas de la 11.1) tendría que fijar de antemano cuántas posiciones admite.
</details>

### 📖 En PyTorch: `nn.RNN`
En la práctica, la memoria no es un número sino un **vector** (por ejemplo, de 16 o 128 neuronas), y las entradas también son vectores (como los *embeddings* de la Lección 11.2). PyTorch hace todas las cuentas por nosotros:

In [ ]:
rnn = nn.RNN(input_size=3, hidden_size=5, batch_first=True)

# Un lote de 2 secuencias, cada una de 4 pasos, cada paso con 3 números
x = torch.randn(2, 4, 3)
salidas, h_final = rnn(x)
print("Entrada        :", tuple(x.shape), "  → (lote, pasos, números por paso)")
print("Salidas        :", tuple(salidas.shape))
print("Memoria final  :", tuple(h_final.shape))
print("Número de pesos:", sum(p.numel() for p in rnn.parameters()))

### 🔮 Ejercicio 4 — Lee las formas
1. `salidas` tiene forma `(2, 4, 5)`. ¿Qué crees que hay en cada uno de esos 4 × 5 números de cada secuencia?
2. ¿Qué relación hay entre `salidas[:, -1]` (la última salida) y `h_final`?
3. 🔮 Si la entrada fuera de 100 pasos en lugar de 4, ¿cuántos pesos tendría la red?

In [ ]:
print("¿Última salida == memoria final?", torch.allclose(salidas[:, -1], h_final[0]))
x_larga = torch.randn(2, 100, 3)
print("Con 100 pasos, salidas:", tuple(rnn(x_larga)[0].shape), "· pesos:", sum(p.numel() for p in rnn.parameters()))

<details><summary><b>👀 Ver respuesta</b></summary>

1. La **memoria en cada paso**: los 5 números del post-it después de leer el paso 1, el 2, el 3 y el 4.
2. Son **iguales**: la memoria final es simplemente la salida del último paso.
3. Los mismos **50** pesos (3 × 5 de entrada + 5 × 5 de memoria + 2 × 5 sesgos). Compartir parámetros en acción.
</details>

---
## 3. 📖 Cómo aprende: retropropagación a través del tiempo

¿Cómo se entrenan los pesos compartidos? Se **desenrolla** la red sobre toda la secuencia y se aplica la retropropagación de siempre (Lección 9.4) sobre ese "tren" de copias. A esto se le llama **retropropagación a través del tiempo** (*BPTT, Backpropagation Through Time*). Como el peso es el mismo en todas las copias, su gradiente es la **suma** de lo que aporta cada paso.

Probémoslo con una tarea donde la RNN brilla: predecir el siguiente valor de una **señal con ritmo** (un sensor que oscila, con ruido).

In [ ]:
rng = np.random.default_rng(0)
t = np.arange(1200)
senal = np.sin(2 * np.pi * t / 20) + 0.1 * rng.normal(size=len(t))      # un ciclo cada 20 pasos

PASOS = 30                                                                 # cuánta historia lee la RNN
X = np.array([senal[i:i + PASOS] for i in range(len(senal) - PASOS)])     # ventanas de la Lección 11.1
y = senal[PASOS:]
X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)                    # (ejemplos, pasos, 1 número por paso)
y = torch.tensor(y, dtype=torch.float32)
corte = 900                                                               # pasado para entrenar, futuro para probar
X_tr, y_tr, X_te, y_te = X[:corte], y[:corte], X[corte:], y[corte:]

plt.plot(senal[:120]); plt.title("Los primeros 120 pasos de la señal"); plt.show()

In [ ]:
class PredictorRNN(nn.Module):
    def __init__(self, oculto=16):
        super().__init__()
        self.rnn = nn.RNN(1, oculto, batch_first=True)
        self.salida = nn.Linear(oculto, 1)              # de la memoria final → un número
    def forward(self, x):
        salidas, _ = self.rnn(x)
        return self.salida(salidas[:, -1]).squeeze(-1)  # usamos solo el último post-it

torch.manual_seed(0)
modelo = PredictorRNN()
opt = torch.optim.Adam(modelo.parameters(), lr=0.01)
for paso in range(300):
    perdida = F.mse_loss(modelo(X_tr), y_tr)
    opt.zero_grad(); perdida.backward(); opt.step()
    if paso % 75 == 0:
        print(f"paso {paso:>3}: pérdida (MSE) = {perdida.item():.4f}")

with torch.no_grad():
    pred = modelo(X_te)
print(f"\nMAE de la RNN en el futuro        : {(pred - y_te).abs().mean():.3f}")
print(f"MAE de 'igual que el paso anterior': {(X_te[:, -1, 0] - y_te).abs().mean():.3f}")

### ✋ Ejercicio 5 — ¿Aprendió?
1. ¿Le gana la RNN a la línea base ingenua? ¿Por cuánto?
2. El ruido que le sumamos a la señal tiene una desviación de 0.1, que equivale a un MAE de ≈0.08. ¿Qué tan cerca está la RNN de ese límite? ¿Podría bajar mucho más?

✍️ *Mis respuestas:*

In [ ]:
plt.plot(y_te[:100].numpy(), label="real")
plt.plot(pred[:100].numpy(), "--", label="predicción RNN")
plt.legend(); plt.title("100 pasos del futuro"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. Sí, y por mucho: la RNN queda en **≈0.08** de MAE, contra **≈0.22** de "igual que el paso anterior" (casi 3 veces menos error).
2. Está prácticamente **en** el límite (≈0.08). El ruido es, por definición, impredecible: ningún modelo puede bajar de ahí. Cuando tu modelo llega al nivel del ruido, **ya terminaste**.

Para esta tarea, a la RNN le basta con recordar **unos pocos pasos** (un ciclo dura 20): su memoria corta no es problema. 👇 Veamos qué pasa cuando sí lo es.
</details>

---
## 4. 📖 Qué pasa si… hay que recordar algo de hace mucho

Recuerda la Sección 2: con `w_h = 0.5`, un recuerdo se multiplica por ~0.5 en cada paso. Durante el entrenamiento, el **gradiente** (la "culpa" del error, Lección 9.4) viaja **hacia atrás** por el mismo camino… y también se multiplica en cada paso.

```
0.5 × 0.5 × 0.5 × … (20 veces)  ≈ 0.000001     → el gradiente DESAPARECE
1.5 × 1.5 × 1.5 × … (20 veces)  ≈ 3,300        → el gradiente EXPLOTA
```

Es el problema de **gradientes que desaparecen o explotan** de la Lección 9.7, pero ahora a lo largo del **tiempo**: una secuencia de 100 pasos es como una red de 100 capas.

### 🔮 Ejercicio 6 — ¿Cuánto "siente" la salida al pasado?
Le pasamos a una RNN una secuencia de 60 pasos y medimos cuánto cambiaría su salida final si moviéramos un poquito la entrada de cada paso (el gradiente de la salida respecto a cada entrada). Antes de ejecutar: ¿cómo crees que será esa sensibilidad para el paso que está **40 pasos atrás**, comparada con la del último paso?

✍️ *Mi predicción:*

In [ ]:
torch.manual_seed(0)
rnn_prueba = nn.RNN(1, 32, batch_first=True)
x = torch.randn(64, 60, 1, requires_grad=True)
salidas, _ = rnn_prueba(x)
salidas[:, -1].abs().sum().backward()               # ¿cuánto depende la salida final de cada entrada?

sensibilidad = x.grad.abs().mean(0)[:, 0].flip(0)   # posición k = "k pasos hacia atrás"
for k in [0, 5, 10, 20, 40]:
    print(f"{k:>2} pasos atrás: sensibilidad = {sensibilidad[k].item():.1e}")

plt.semilogy(sensibilidad.numpy())
plt.xlabel("pasos hacia atrás"); plt.ylabel("sensibilidad (escala log)")
plt.title("La memoria de una RNN se apaga exponencialmente"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

La sensibilidad cae en picada: de **1** en el último paso a ≈**3e-2** a 5 pasos, ≈**2e-3** a 10, ≈**8e-6** a 20 y ≈**3e-10** a 40 pasos. En escala logarítmica es casi una **línea recta hacia abajo**: caída **exponencial**.

En la práctica, lo que pasó hace 40 pasos **no influye en nada**, y el entrenamiento no recibe ninguna señal de que debería recordarlo. La RNN "no sabe que debía acordarse".
</details>

### 🔮 Ejercicio 7 — La prueba de memoria
Diseñamos una tarea que **solo** se resuelve recordando el pasado lejano:
- En el **primer paso** la red ve un número secreto entre −1 y 1.
- Luego ve **ruido** durante muchos pasos.
- Al final debe decir **cuál era el número secreto**.

Si la red no recuerda nada, lo mejor que puede hacer es decir siempre 0, con un error medio de **0.5**. Si recuerda bien, el error será casi 0.

Entrenaremos la misma RNN con secuencias de **5** pasos y de **25** pasos. ¿Qué error esperas en cada caso?

✍️ *Mi predicción:*

In [ ]:
def lote_memoria(n, largo, generador):
    x = torch.zeros(n, largo, 2)
    secreto = torch.rand(n, generator=generador) * 2 - 1
    x[:, 0, 0] = secreto                                              # canal 0: el secreto, solo en el paso 1
    x[:, 1:, 1] = torch.randn(n, largo - 1, generator=generador)     # canal 1: ruido que distrae
    return x, secreto

class Memorion(nn.Module):
    def __init__(self, tipo="RNN", oculto=32):
        super().__init__()
        self.red = getattr(nn, tipo)(2, oculto, batch_first=True)
        self.salida = nn.Linear(oculto, 1)
    def forward(self, x):
        salidas, _ = self.red(x)
        return self.salida(salidas[:, -1]).squeeze(-1)

def prueba_de_memoria(tipo, largo, pasos=400, semilla=0):
    torch.manual_seed(semilla); gen = torch.Generator().manual_seed(semilla)
    modelo = Memorion(tipo)
    opt = torch.optim.Adam(modelo.parameters(), lr=0.005)
    for _ in range(pasos):
        x, secreto = lote_memoria(64, largo, gen)
        perdida = F.mse_loss(modelo(x), secreto)
        opt.zero_grad(); perdida.backward()
        nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)          # recorte de gradiente (Sección 5)
        opt.step()
    x, secreto = lote_memoria(1000, largo, gen)
    with torch.no_grad():
        return (modelo(x) - secreto).abs().mean().item()

for largo in [5, 25]:
    print(f"RNN, secreto {largo:>2} pasos atrás → error medio = {prueba_de_memoria('RNN', largo):.3f}   (no recordar nada = 0.5)")

<details><summary><b>👀 Ver respuesta</b></summary>

- **5 pasos:** error ≈ **0.01–0.02**: recuerda el secreto casi perfecto.
- **25 pasos:** error ≈ **0.25** (y con otras semillas puede quedarse en ≈0.5, como si no recordara nada). Con el mismo entrenamiento, la RNN **no logra** llevar el secreto 25 pasos.

25 pasos no es nada: una frase normal tiene 20–30 tokens. Si la pregunta depende del sujeto que apareció al inicio ("**Los niños** que vinieron ayer a la fiesta de cumpleaños de mi prima … **estaban** cansados"), una RNN simple lo pierde. La solución, en la Lección 11.4: **puertas** que deciden qué recordar y qué olvidar.
</details>

---
## 5. 📖 Hiperparámetros y perillas

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `hidden_size` | Tamaño del post-it (memoria) | Recuerda más cosas a la vez; más lento y más riesgo de sobreajuste | Memoria pobre | 32–256 |
| `num_layers` | RNN apiladas (la salida de una es la entrada de la otra) | Patrones más abstractos; más difícil de entrenar | Más simple | 1–2 |
| Largo de la secuencia (BPTT) | Cuántos pasos se desenrollan para entrenar | Puede aprender dependencias más largas; más memoria y más lento | Entrena rápido, pero "no ve" lo lejano | Lo que necesite la tarea (20–200) |
| Recorte de gradiente (`clip_grad_norm_`) | Tope al tamaño del gradiente | Menos protección contra explosiones | Pasos muy tímidos | 1.0–5.0 |
| `bidirectional` | Leer también de derecha a izquierda | Usa contexto del futuro (no sirve para predecir el futuro) | Solo pasado | `False` para pronóstico; `True` para clasificar textos completos |
| Tasa de aprendizaje | Tamaño de los pasos (Lección 3.4) | Inestable, explota | Lento | 1e-3 (Adam) |

> 💡 **Recorte de gradiente:** es el "cinturón de seguridad" de las RNN. Si el gradiente **explota**, lo recorta a un tamaño máximo antes de dar el paso. No arregla los gradientes que **desaparecen** (no se puede agrandar lo que vale 0).

### ✋ Ejercicio 8
Tu RNN entrena bien unos pasos y de repente la pérdida salta a `nan`. ¿Qué perilla tocas primero? ¿Y si la pérdida baja bien, pero el modelo ignora todo lo que pasó al principio de cada secuencia?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **Pérdida `nan`** → gradientes que **explotan**: añade o baja el **recorte de gradiente** (y/o baja la tasa de aprendizaje). Lo vimos en la Lección 9.9.
- **Ignora el principio** → gradientes que **desaparecen**: subir `hidden_size` o recortar el gradiente **no** lo arregla. Hay que cambiar de arquitectura: **LSTM o GRU** (Lección 11.4) o, mejor aún, atención (Módulo 12).
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Pérdida que salta a `nan` o picos enormes | Gradientes que explotan | `clip_grad_norm_`, bajar la tasa de aprendizaje |
| El modelo ignora lo que pasó hace muchos pasos | Gradientes que desaparecen (memoria corta) | LSTM/GRU (11.4), atención (M12) |
| Error de formas: esperaba `(lote, pasos, rasgos)` | Olvidaste `batch_first=True` (por defecto PyTorch usa `(pasos, lote, rasgos)`) | Revisa `batch_first` y `x.shape` |
| Predice bien en validación pero "hace trampa" | Barajaste la serie o usaste `bidirectional=True` para pronosticar | Validación temporal (11.1); RNN en una sola dirección |
| Entrenamiento muy lento con secuencias largas | Cada paso depende del anterior: **no se paraleliza** | Secuencias más cortas, o Transformers (M12) |
| El texto generado repite lo mismo o copia el entrenamiento | Temperatura muy baja o sobreajuste | Subir temperatura, menos épocas (Sección 9.3) |

### 🔮 Ejercicio 9
Una RNN procesa una frase de 50 tokens. Un Transformer (M12) procesa todos los tokens **a la vez**. Si tienes una GPU con miles de núcleos, ¿cuál de los dos entrenará más rápido y por qué?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

El **Transformer**. La RNN es inherentemente **secuencial**: no puede calcular h₅₀ sin h₄₉, ni h₄₉ sin h₄₈… Aunque tengas miles de núcleos, 49 de cada 50 están esperando. El Transformer calcula todas las posiciones en paralelo. Esta, junto con la memoria corta, es la razón por la que los Transformers reemplazaron a las RNN en lenguaje.
</details>

---
## 7. 📋 Ficha del modelo

| | Red recurrente simple (RNN, "vanilla RNN") |
|---|---|
| **Tipo** | Red neuronal para secuencias: clasificar, predecir el siguiente paso, generar |
| **Intuición** | Lee paso a paso y reescribe un post-it (estado oculto) con lo que recuerda |
| **Cómo aprende** | Retropropagación a través del tiempo (BPTT) sobre la red desenrollada |
| **Hiperparámetros clave** | `hidden_size`, `num_layers`, largo de secuencia, recorte de gradiente |
| **✅ Fortalezas** | Secuencias de cualquier longitud · Pocos parámetros (compartidos) · Simple · Generación paso a paso |
| **❌ Debilidades** | Memoria **corta** (gradientes que desaparecen) · Gradientes que explotan · Lenta: no se paraleliza |
| **Úsalo cuando…** | Secuencias cortas, dispositivos con poca memoria, o para entender la base de LSTM/GRU |
| **Evítalo cuando…** | Hay dependencias largas (usa LSTM/GRU) o mucho texto (usa Transformers) |

## 8. 🧠 Resumen en 5 puntos
1. Una **RNN** lee la secuencia **paso a paso** y lleva una **memoria** (estado oculto) que reescribe en cada paso: hₜ = tanh(wₓ·xₜ + wₕ·hₜ₋₁ + b).
2. Usa **los mismos pesos** en todos los pasos: por eso acepta secuencias de cualquier longitud con pocos parámetros.
3. Se entrena con **retropropagación a través del tiempo**: la red desenrollada es como una red muy profunda.
4. El gradiente se **multiplica** en cada paso: se **desvanece** (memoria corta) o **explota** (`nan`). El recorte de gradiente arregla lo segundo, no lo primero.
5. Funciona muy bien cuando basta con recordar **pocos pasos** (señales con ritmo, nombres cortos), pero falla con dependencias largas: motivación de **LSTM y GRU** (11.4).

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** En una RNN, ¿qué representa el estado oculto hₜ?
- a) La predicción final · b) Un resumen de todo lo leído hasta el paso *t* · c) El peso de la red · d) El error

**2.** Una RNN con `hidden_size=64` procesa una frase de 10 palabras y luego otra de 1,000. ¿Cuántos pesos usa en cada caso?
- a) 10 veces más en la larga · b) 100 veces más en la larga · c) Los mismos · d) Depende de la GPU

**3.** La pérdida de tu RNN se vuelve `nan` en la época 3. ¿Qué es lo **primero** que pruebas?
- a) Más neuronas · b) Recorte de gradiente y/o tasa de aprendizaje más baja · c) Más épocas · d) `bidirectional=True`

**4.** ¿Por qué una RNN simple olvida lo que pasó hace muchos pasos?
- a) Porque el gradiente se multiplica muchas veces por números pequeños y se desvanece · b) Porque se le acaba la RAM · c) Porque la `tanh` es lineal · d) Porque usa demasiados pesos

**5.** Quieres **predecir** la demanda de mañana con una RNN. ¿Usarías `bidirectional=True`?
- a) Sí, siempre ayuda · b) No: leer "de derecha a izquierda" significa usar el futuro, que no tendrás al predecir

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Es el post-it: la memoria de lo leído.
2. **c)** Los pesos se comparten entre pasos.
3. **b)** `nan` = gradientes que explotan.
4. **a)** Gradientes que desaparecen a través del tiempo.
5. **b)** La bidireccionalidad es para tareas donde ya tienes la secuencia completa (clasificar una reseña), no para pronosticar.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** En la Lección 11.1 usamos ventanas de tamaño fijo con una regresión lineal. Da **una ventaja** y **una desventaja** de la RNN frente a ese enfoque.

**B. Caso de análisis 💼** Un banco quiere detectar fraude leyendo **toda la historia** de movimientos de cada cliente (a veces miles de movimientos) con una RNN simple. El patrón de fraude típico: un pequeño "cargo de prueba" seguido, **varios cientos de movimientos después**, de un cargo grande. ¿Qué problemas anticipas y qué propondrías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Ventaja: la RNN acepta secuencias de **cualquier longitud** y aprende patrones que valen en cualquier posición (comparte pesos). Desventaja: es **más difícil de entrenar** (gradientes que desaparecen/explotan), más lenta y menos interpretable; una regresión sobre ventanas es rápida y sus pesos se leen (Ejercicio 6 de la 11.1).

**B.** El cargo de prueba está cientos de pasos antes del cargo grande: la RNN simple **lo habrá olvidado** (memoria corta) y además entrenar con miles de pasos será lento y propenso a explosiones. Propuestas: (1) LSTM/GRU (11.4) o atención (M12); (2) **ingeniería de características**: resumir la historia en variables como "¿hubo un cargo menor a $10 en los últimos 30 días?", que cualquier modelo de tabla puede usar; (3) recortar la historia a una ventana relevante.
</details>

### 9.3 🧩 Práctica integradora: una RNN que inventa nombres

Vas a entrenar un **modelo de lenguaje de caracteres**: una RNN que lee un nombre letra por letra y aprende a **predecir la siguiente letra**. Luego la usarás para **inventar** nombres nuevos. Es, en miniatura, lo mismo que hace GPT con tokens.

**Paso 1 ▶️ — Los datos.** 180 nombres en español. Usamos el punto `.` como símbolo especial de **inicio** y **fin** de nombre.

In [ ]:
NOMBRES = """
ana andrea adriana alejandra alicia alma amalia amparo angela antonia araceli ariana
aurora beatriz blanca brenda camila carla carmen carolina catalina cecilia claudia clara
cristina daniela debora diana dolores elena elisa elvira emilia esperanza estela eva
fabiola fernanda flor gabriela gloria graciela guadalupe ines irene isabel jimena josefina
juana julia julieta karla laura leticia liliana lorena lucia luisa luz magdalena manuela
marcela margarita maria mariana marina marta mercedes micaela miriam monica natalia noemi
norma olga paola patricia paula pilar raquel rebeca regina rocio rosa rosario sandra sara
silvia sofia soledad susana teresa valentina valeria vanesa veronica victoria ximena
yolanda adrian agustin alberto alejandro alfonso alfredo alvaro andres angel antonio
armando arturo benjamin bernardo carlos cesar cristian daniel david diego eduardo emilio
enrique ernesto esteban fabian federico felipe fernando francisco gabriel gerardo gonzalo
guillermo gustavo hector hugo ignacio ivan jaime javier jesus joaquin jorge jose juan
julian julio leonardo lorenzo lucas luis manuel marco mario martin mateo mauricio miguel
nicolas oscar pablo pedro rafael ramiro raul ricardo roberto rodrigo rolando ruben
salvador samuel santiago sebastian sergio simon tomas ulises vicente victor
""".split()

alfabeto = ["."] + sorted(set("".join(NOMBRES)))        # "." = inicio/fin
c2i = {c: i for i, c in enumerate(alfabeto)}
V = len(alfabeto)

def codificar(nombre):
    return [c2i[c] for c in "." + nombre + "."]

print(f"{len(NOMBRES)} nombres · {V} símbolos: {''.join(alfabeto)}")
print("'ana' codificado:", codificar("ana"))

✋ La RNN recibe `. a n a` y debe predecir `a n a .` (lo mismo, corrido una posición). ¿Qué tiene que predecir **después de ver `. a`**? ¿Hay una sola respuesta correcta?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Debe predecir `n` (para "ana"), pero en los datos también hay "andrea", "alma", "adrián", "alberto"… Después de `. a` hay **muchas** letras posibles. Por eso la red no predice **una** letra, sino una **probabilidad** para cada letra. Al generar, **sortearemos** la siguiente letra según esas probabilidades.
</details>

**Paso 2 ▶️ — La red y el entrenamiento.** Un *embedding* por letra (Lección 11.2) → RNN → una capa que puntúa cada letra posible.

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class RNNdeNombres(nn.Module):
    def __init__(self, oculto=64):
        super().__init__()
        self.emb = nn.Embedding(V, 16)
        self.rnn = nn.RNN(16, oculto, batch_first=True)
        self.salida = nn.Linear(oculto, V)
    def forward(self, x, h=None):
        salidas, h = self.rnn(self.emb(x), h)
        return self.salida(salidas), h

# Todas las secuencias del mismo largo, rellenando con -1 (padding, Lección 11.1)
datos = pad_sequence([torch.tensor(codificar(n)) for n in NOMBRES], batch_first=True, padding_value=-1)
entrada, objetivo = datos[:, :-1].clamp(min=0), datos[:, 1:]       # objetivo = la letra siguiente

def entrenar_nombres(epocas, oculto=64, semilla=0):
    torch.manual_seed(semilla)
    red = RNNdeNombres(oculto)
    opt = torch.optim.Adam(red.parameters(), lr=0.005)
    for _ in range(epocas):
        puntajes, _ = red(entrada)
        perdida = F.cross_entropy(puntajes.reshape(-1, V), objetivo.reshape(-1), ignore_index=-1)  # ignora el relleno
        opt.zero_grad(); perdida.backward(); opt.step()
    print(f"{epocas} épocas → pérdida final {perdida.item():.2f}")
    return red

red = entrenar_nombres(epocas=150)

**Paso 3 ▶️ — ¡A inventar!** Para generar: empezamos con `.`, le pedimos a la red las probabilidades de la siguiente letra, **sorteamos** una, se la damos como nueva entrada y repetimos hasta que salga `.` (fin).

La **temperatura** controla el sorteo (la verás a fondo en la Lección 13.6):
- **Baja** (0.3): casi siempre elige la letra más probable → nombres "seguros" y repetidos.
- **Alta** (1.5): sortea casi al azar → nombres raros, a veces impronunciables.

In [ ]:
@torch.no_grad()
def inventar(red, temperatura=1.0, maximo=15):
    x, h, nombre = torch.tensor([[0]]), None, ""
    for _ in range(maximo):
        puntajes, h = red(x, h)                                          # la memoria h pasa de letra en letra
        probas = torch.softmax(puntajes[0, -1] / temperatura, dim=-1)
        i = torch.multinomial(probas, 1).item()                          # sorteo
        if i == 0: break                                                 # salió "." → fin del nombre
        nombre += alfabeto[i]
        x = torch.tensor([[i]])
    return nombre

def reporte(red, temperatura, n=200):
    torch.manual_seed(1)
    inventados = [inventar(red, temperatura) for _ in range(n)]
    nuevos = sum(nombre not in NOMBRES for nombre in inventados) / n
    print(f"T={temperatura}: {nuevos:.0%} no están en la lista · ejemplos: {inventados[:8]}")

for T in [0.5, 1.0, 1.5]:      # ✏️ prueba otras temperaturas
    reporte(red, T)

🔮 ¿Qué pasa con el **porcentaje de nombres nuevos** al subir la temperatura? ¿Y con lo **pronunciables** que son?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Al subir la temperatura sube el porcentaje de nombres nuevos (aprox. **≈30 %** con T=0.5, **≈75 %** con T=1.0 y **≈90 %** con T=1.5), pero con T=1.5 aparecen engendros como "hedjobuas" o "gapcimo". Con T baja salen muchos nombres reales de la lista (maría, antonio, rosario…). Es el clásico compromiso **creatividad vs. coherencia**. (Los nombres exactos pueden variar en tu máquina; el patrón no.)
</details>

**Paso 4 🔮 — ¿Y si entrenamos más?** Vamos a entrenar el **doble** (300 épocas). La pérdida bajará… ¿los nombres inventados serán **mejores**?

✍️ *Mi predicción:*

In [ ]:
red_300 = entrenar_nombres(epocas=300)       # ✏️ prueba también 80
reporte(red_300, 1.0)

<details><summary><b>👀 Ver respuesta</b></summary>

La pérdida baja (de ≈0.98 a ≈0.75), pero con T=1.0 solo **≈20 %** de los nombres son nuevos (antes ≈75 %): la red **memorizó la lista** y casi solo repite nombres de entrenamiento. ¡Es **sobreajuste** (Lección 4.4)! Para "inventar" queríamos que aprendiera *cómo suenan* los nombres en español, no la lista exacta. Con 80 épocas pasa lo contrario: casi todo es nuevo, pero muchos nombres suenan raros (subajuste).

Moraleja: en generación, una pérdida de entrenamiento más baja **no siempre** es mejor. Por eso los modelos de lenguaje se evalúan con texto **que no vieron**.
</details>

**Paso 5 ✋ — Piensa en la memoria.** Esta RNN genera nombres de 4 a 10 letras sin problema. ¿Crees que podría generar un **párrafo coherente** de 500 letras? ¿Por qué?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Muy difícil. Para que un párrafo sea coherente hay que recordar de qué se hablaba hace cientos de letras, y ya viste que la memoria de una RNN simple se apaga en unas decenas de pasos. Generaría palabras que "suenan" bien localmente, pero sin hilo. Las LSTM (Lección 11.4) mejoraron mucho esto (fueron el estado del arte en generación de texto hacia 2015), y los Transformers (M12) lo resolvieron a gran escala.
</details>

---
## 📚 Para profundizar
- 📘 **Artículo clásico:** Andrej Karpathy, *The Unreasonable Effectiveness of Recurrent Neural Networks* (2015): RNN de caracteres que escriben Shakespeare y código.
- 📺 **Video:** Andrej Karpathy, *makemore* (parte 1 y siguientes): un modelo de nombres desde cero, paso a paso (inglés).
- 📺 **Video:** StatQuest, "Recurrent Neural Networks (RNNs), Clearly Explained" (inglés, con subtítulos).

---
⬅️ **Anterior:** [11.2 Representar palabras: tokens y *embeddings* (el mapa de significados)](11.2-tokens-y-embeddings.ipynb) · ➡️ **Siguiente:** [11.4 LSTM y GRU: puertas para recordar y olvidar](11.4-lstm-y-gru.ipynb)